Análisis Exploratorio

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configuración para mejorar la visualización de las gráficas
plt.style.use('seaborn-v0_8_whitegrid') # Estilo de gráficas
sns.set_palette("viridis") # Paleta de colores

# --- 1. Carga de Datos ---
# Asegúrate de que la ruta al archivo CSV sea correcta.
# Si el script está en la misma carpeta que la carpeta 'CSVs':
file_path = os.path.join('CSVs', 'Air_Quality.csv')
# Si el script está DENTRO de la carpeta 'CSVs':
# file_path = 'Air_Quality.csv'

try:
    df = pd.read_csv(file_path)
except FileNotFoundError:
    print(f"Error: El archivo '{file_path}' no fue encontrado. Verifica la ruta.")
    exit()

print("--- Primeras filas del DataFrame ---")
print(df.head())
print("\n" + "="*50 + "\n")

# Convertir la columna 'Date' a datetime y manejar errores de zona horaria
# El formato '+00:00' indica UTC. Pandas usualmente lo maneja bien,
# pero si da problemas, podemos hacerlo explícito o quitarlo.
try:
    df['Date'] = pd.to_datetime(df['Date'])
except Exception as e:
    print(f"Error al convertir 'Date': {e}")
    # Intento alternativo si hay problemas con la zona horaria (ej. quitándola)
    # df['Date'] = pd.to_datetime(df['Date'].str.replace(r'[+-]\d{2}:\d{2}$', '', regex=True), errors='coerce')
    # df['Date'] = df['Date'].dt.tz_localize(None) # Si quieres quitar la info de zona horaria

# Establecer 'Date' como índice si es apropiado para análisis de series temporales,
# aunque para algunas visualizaciones es más fácil tenerla como columna.
# Por ahora, la mantenemos como columna para facilitar el agrupamiento por 'City'.

# --- 2. Información Básica y Estadísticas Descriptivas ---
print("--- Información del DataFrame ---")
df.info()
print("\n" + "="*50 + "\n")

print("--- Estadísticas Descriptivas (Variables Numéricas) ---")
# Excluimos 'Date' si aún no es índice y 'City'
numeric_cols = df.select_dtypes(include=['number']).columns
print(df[numeric_cols].describe().T)
print("\n" + "="*50 + "\n")

print("--- Estadísticas Descriptivas de AQI por Ciudad ---")
print(df.groupby('City')['AQI'].describe().T)
print("\n" + "="*50 + "\n")

# --- 3. Análisis de Valores Faltantes ---
print("--- Conteo de Valores Faltantes por Columna ---")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])
print("\n" + "="*50 + "\n")

# Visualización de valores faltantes (heatmap)
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Mapa de Calor de Valores Faltantes')
plt.tight_layout()
plt.show()

# --- 4. Distribución de las Variables Numéricas ---
# Seleccionar solo columnas numéricas, excluyendo 'Date' y 'City' si es necesario
# (ya lo hicimos con numeric_cols)

# Histogramas
print("--- Generando Histogramas de Variables Numéricas ---")
df[numeric_cols].hist(bins=30, figsize=(15, 10), layout=(-1, 3))
plt.suptitle('Histogramas de Variables Numéricas', y=1.02, size=16)
plt.tight_layout(rect=[0, 0, 1, 0.98]) # Ajuste para el supertítulo
plt.show()

# Boxplots por variable (para ver outliers y distribución general)
print("--- Generando Boxplots de Variables Numéricas ---")
plt.figure(figsize=(18, 10))
for i, col in enumerate(numeric_cols):
    plt.subplot(2, (len(numeric_cols) + 1) // 2, i + 1)
    sns.boxplot(y=df[col])
    plt.title(col)
    plt.ylabel('')
plt.suptitle('Boxplots de Variables Numéricas', y=1.02, size=16)
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

# Boxplots de cada contaminante y AQI, agrupados por Ciudad
print("--- Generando Boxplots de Variables Numéricas por Ciudad ---")
for col in numeric_cols: # Iteramos sobre todas las numéricas, incluyendo AQI
    if col == 'CO2' and df[col].isnull().all(): # Omitir CO2 si está completamente vacío
        print(f"Omitiendo boxplot para {col} ya que todos sus valores son nulos.")
        continue
    plt.figure(figsize=(12, 7))
    sns.boxplot(x='City', y=col, data=df)
    plt.title(f'Distribución de {col} por Ciudad')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

# --- 5. Análisis Específico de la Variable Objetivo 'AQI' ---
print("--- Análisis de la Variable Objetivo AQI ---")
plt.figure(figsize=(10, 6))
sns.histplot(df['AQI'], kde=True, bins=30)
plt.title('Distribución del AQI (Todas las Ciudades)')
plt.xlabel('AQI')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()

# --- 6. Series Temporales de AQI por Ciudad ---
print("--- Generando Gráficos de Series Temporales de AQI por Ciudad ---")
cities = df['City'].unique()
num_cities = len(cities)

# Determinar el layout de subplots
n_cols = 2
n_rows = (num_cities + n_cols - 1) // n_cols

plt.figure(figsize=(15, n_rows * 4))
for i, city in enumerate(cities):
    plt.subplot(n_rows, n_cols, i + 1)
    city_data = df[df['City'] == city].set_index('Date')
    city_data['AQI'].plot(legend=False)
    plt.title(f'AQI en {city}')
    plt.xlabel('Fecha')
    plt.ylabel('AQI')
    plt.xticks(rotation=30, ha='right')
plt.suptitle('Evolución Temporal del AQI por Ciudad', y=1.02, size=16)
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

# Plot de AQI de todas las ciudades en una misma gráfica (puede ser denso)
plt.figure(figsize=(15, 7))
for city in cities:
    city_data = df[df['City'] == city].set_index('Date')
    # Usar un rolling mean para suavizar si las series son muy ruidosas
    # city_data['AQI'].rolling(window=24).mean().plot(label=city, alpha=0.8) # Ejemplo con suavizado diario
    city_data['AQI'].plot(label=city, alpha=0.7)
plt.title('Evolución Temporal del AQI (Todas las Ciudades)')
plt.xlabel('Fecha')
plt.ylabel('AQI')
plt.legend(title='Ciudad', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout(rect=[0, 0, 0.85, 1]) # Ajuste para la leyenda fuera
plt.show()


# --- 7. Matriz de Correlación ---
print("--- Generando Matriz de Correlación (Variables Numéricas) ---")
# Seleccionar solo las columnas numéricas para la correlación
correlation_matrix = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Matriz de Correlación de Variables Numéricas')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Correlación con AQI específicamente
print("\n--- Correlación de Variables con AQI ---")
corr_with_aqi = correlation_matrix['AQI'].sort_values(ascending=False)
print(corr_with_aqi)
print("\n" + "="*50 + "\n")

# Correlación por ciudad (Ejemplo para una ciudad, puedes iterar)
if len(cities) > 0:
    sample_city = cities[0]
    print(f"--- Matriz de Correlación para {sample_city} ---")
    city_corr = df[df['City'] == sample_city][numeric_cols].corr()
    plt.figure(figsize=(10, 8))
    sns.heatmap(city_corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
    plt.title(f'Matriz de Correlación para {sample_city}')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()


# --- 8. Información adicional sobre los datos ---
print("--- Rangos de Fechas por Ciudad ---")
for city in cities:
    city_data = df[df['City'] == city]
    min_date = city_data['Date'].min()
    max_date = city_data['Date'].max()
    count = len(city_data)
    print(f"Ciudad: {city}, Registros: {count}, Desde: {min_date}, Hasta: {max_date}")
print("\n" + "="*50 + "\n")

print("--- Conteo de Registros por Ciudad ---")
print(df['City'].value_counts())
print("\n" + "="*50 + "\n")

print("Análisis exploratorio básico completado.")